In [1]:
import sys
import os
import pandas as pd

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [4]:
import yaml
from pathlib import Path

from data_scraping.common.data_loader import load_ratings_data
from modeling.utils.data import filter_by_min_counts

"""Item-Based CF 파이프라인 실행"""

# 데이터 설정 로드
data_config_path = "/Users/visuworks/Desktop/movie_recommendation/modeling/utils/data_config.yaml"
print(f"📄 데이터 설정 파일 로드: {data_config_path}")
with open(data_config_path, 'r', encoding='utf-8') as f:
    data_config_dict = yaml.safe_load(f)


data_config = data_config_dict['data']
min_user_ratings = data_config.get('min_user_ratings', 10)
min_movie_ratings = data_config.get('min_movie_ratings', 30)

print("\n" + "="*60)
print("🎬 Item-Based Collaborative Filtering 파이프라인")
print("="*60)
        
print("📥 데이터를 로드하는 중...")
df_ratings = load_ratings_data()
print(f"  - 데이터: {len(df_ratings):,}개 평점")

print("🔍 데이터를 필터링하는 중...")
print(f"  - 필터링 조건: 사용자당 최소 {min_user_ratings}개, 영화당 최소 {min_movie_ratings}개")            
filtered_data = filter_by_min_counts(df_ratings, min_movie_ratings=min_movie_ratings, min_user_ratings=min_user_ratings)
print(f"  - 필터링된 데이터: {len(filtered_data):,}개 평점")

📄 데이터 설정 파일 로드: /Users/visuworks/Desktop/movie_recommendation/modeling/utils/data_config.yaml

🎬 Item-Based Collaborative Filtering 파이프라인
📥 데이터를 로드하는 중...
  - 데이터: 31,242,048개 평점
🔍 데이터를 필터링하는 중...
  - 필터링 조건: 사용자당 최소 10개, 영화당 최소 30개

[필터링 전] 사용자 수: 200,945명, 영화 수: 76,670개
[필터 적용] min_user_ratings: 10, min_movie_ratings: 30
조건 통과 사용자 수: 200,932명, 조건 통과 영화 수: 18,514개
[필터링 후] 사용자 수: 200,932명, 영화 수: 18,514개
필터링된 평점 수: 30,914,589개 (제거: 327,459개)
  - 필터링된 데이터: 30,914,589개 평점


In [ ]:
# 3. 추천 시스템 초기화 및 학습
recommender = ItemBasedRecommender(config=config)
recommender.fit(filtered_data)

# 3. 모델 저장
save_path = Path(__file__).parent / 'models' / 'pkls' / 'trained_item_based.pkl'
recommender.save(save_path)

# 4. 추천 테스트
print("\n" + "="*60)
print("🎬 추천 테스트")
print("="*60)

# 첫 번째 movie_id로 추천 테스트
first_movie_id = filtered_data['movie_id'].iloc[0]
print(f"테스트할 영화 ID: {first_movie_id}")

recommendations = recommender.recommend(
    movie_id=first_movie_id,
    top_n=5,
    return_scores=True
)

if recommendations is not None:
    display_cols = ['movie_id', 'title', 'similarity_score']
    print("\n추천 결과:")
    print(recommendations[display_cols].to_string(index=False))
else:
    print(f"경고: 영화 ID '{first_movie_id}'에 대한 추천을 찾을 수 없습니다.")

# 5. 모델 로드 테스트
print("\n\n" + "="*60)
print("📂 모델 로드 테스트")
print("="*60)

loaded_recommender = ItemBasedRecommender.load(save_path)
print("✅ 로드된 모델로 추천 테스트 완료!")

print("\n" + "="*60)
print("✅ Item-Based CF 파이프라인 완료!")
print("="*60)
print(f"\n💾 저장된 모델 위치: {save_path}")
print("\n사용 예시:")
print("  from models.item_based import ItemBasedRecommender")
print(f"  recommender = ItemBasedRecommender.load('{save_path}')")
print("  recommendations = recommender.recommend_by_title('영화제목', top_n=10)")